<a href="https://colab.research.google.com/github/sunny-coder-singh/First/blob/main/Major_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pandas faiss-cpu sentence-transformers transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.6 MB/s eta 0:00:00


In [6]:
pip install streamlit pandas numpy faiss-cpu sentence-transformers transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 64.5 MB/s eta 0:00:00


In [12]:
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/New Project/Starbucks_Synthetic_Beverage_Dataset_3000.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import pandas as pd
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ==========================
# Load Coffe Dataset
# ==========================
df = pd.read_csv("/content/drive/MyDrive/New Project/Starbucks_Synthetic_Beverage_Dataset_3000.csv")

print(f"Rows    : {len(df)}")
print(f"Columns : {list(df.columns)}")

# Fill missing values
df = df.fillna("Unknown")

# Convert each row into text
documents = df.astype(str).apply(
    lambda row: " | ".join(row),
    axis=1
).tolist()

# ==========================
# Load Embedding Model
# ==========================
print("\nLoading embedding model...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# ==========================
# Build FAISS Index
# ==========================
dimension = embeddings.shape[1]

# Cosine Similarity Index
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Indexed {index.ntotal} documents")

# ==========================
# Load LLM
# ==========================
device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=device
)

print("LLM Loaded")

# ==========================
# Chat Function
# ==========================
def ask_bot(question, top_k=5):

    if not question.strip():
        return "Please enter a valid question."

    # Encode question
    q_embedding = embedder.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Search similar rows
    scores, indices = index.search(q_embedding, top_k)

    if scores[0][0] < 0.30:
        return "I could not find relevant information in the Wine dataset."

    context = "\n".join(
        documents[i] for i in indices[0]
    )

    prompt = f"""
You are a Starbucks Dataset Assistant.

Answer ONLY using the information provided in the dataset.

Rules:
- Do not make up information.
- If the answer is not available in the dataset, reply:
  "The dataset does not contain this information."
- Keep answers short and accurate.

Dataset:
{context}

Question:
{question}

Answer:
"""

    try:
        output = generator(
            prompt,
            max_new_tokens=120,
            do_sample=False
        )

        return output[0]["generated_text"].strip()

    except Exception as e:
        return f"Error: {e}"

# ==========================
# Chat Loop
# ==========================
print("\n🍷 Starbucks Dataset Chatbot Ready!")
print("Type 'exit' to quit.\n")

while True:

    question = input("You: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = ask_bot(question)

    print("\nBot:", answer)

Rows    : 3000
Columns : ['Beverage_ID', 'Beverage_Name', 'Category', 'Size', 'Hot_Cold', 'Ingredients', 'Milk_Type', 'Espresso_Shots', 'Syrup', 'Toppings', 'Sweetness_Level', 'Caffeine_mg', 'Calories', 'Sugar_g', 'Protein_g', 'Fat_g', 'Price_USD', 'Price_INR', 'Customer_Rating', 'Number_of_Orders', 'Popularity', 'Seasonal', 'Vegan', 'Vegetarian', 'Contains_Dairy', 'Contains_Nuts', 'Gluten_Free', 'Recommended_With', 'Best_Time', 'Flavor_Profile', 'Temperature', 'Country_Availability']

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Indexed 3000 documents


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

LLM Loaded

🍷 Starbucks Dataset Chatbot Ready!
Type 'exit' to quit.

You: Mango Refresher


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (668 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: You are a Starbucks Dataset Assistant.

Answer ONLY using the information provided in the dataset.

Rules:
- Do not make up information.
- If the answer is not available in the dataset, reply:
  "The dataset does not contain this information."
- Keep answers short and accurate.

Dataset:
SB00523 | Mango Refresher | Refresher | Grande | Cold | A handcrafted Mango Refresher prepared with premium Starbucks-style coffee or tea, whole milk, toffee nut syrup and carefully balanced flavors. | Whole | 3 | Caramel | Whipped Cream | Medium | 191 | 113 | 48 | 2.7 | 14.9 | 7.19 | 611.15 | 3.8 | 47781 | Very High | No | Yes | Yes | Yes | No | Yes | Croissant | Afternoon | Sweet | Cold | Japan
SB00005 | Mango Refresher | Refresher | Venti | Cold | A handcrafted Mango Refresher prepared with premium Starbucks-style coffee or tea, coconut milk, none syrup and carefully balanced flavors. | Almond | 1 | Toffee Nut | Whipped Cream | High | 80 | 298 | 22 | 4.0 | 28.4 | 5.87 | 498.95 | 4.1 | 13354 | 